# 04. (실험) LLM API 없이 측정하는 De-Fluff 파이프라인

> **상태: 실험 중.** 아직 전체 데이터셋에 적용한 결과는 없습니다.

`01`의 Gemini 기반 파이프라인은 비용·속도와 평가 모델 편향 문제가 있어, 공개 로컬 모델만으로 같은 일을 하는 방법을 시도한 노트북입니다.

| 단계 | 방법 |
|---|---|
| 1. 압축 후보 생성 | TextRank · KL-Sum(추출), LLMLingua-2(토큰 제거), BART-large-CNN(추상 요약) |
| 2. Fluff 측정 | Hyland(2005) interactional metadiscourse 표지 밀도(100단어당) + Yule's K |
| 3. 정보 보존 | Nugget 평가: DeBERTa NLI로 핵심 행동(nugget) 함의 여부 판정, vital recall / F-beta |
| 4. 사실성 | SummaC(Conv) 일관성·보존 점수 (선택: AlignScore) |
| 선택 | 제약(vital recall=1, 전체 recall≥0.9, SummaC≥0.2, 길이≥25%)을 만족하는 후보 중 metadiscourse 밀도 최소 |

In [ ]:
# 필요한 패키지 설치
!pip install -q -U transformers torch llmlingua
# SummaC는 선택 사항입니다. 설치·로딩에 실패하면 사실성 검사는 자동으로 건너뜁니다.
!pip install -q summac

## 설정 · 공통 유틸리티

In [ ]:
import os
import re
import math
import json
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COUNT_TOKENIZER = "Qwen/Qwen2.5-7B-Instruct"
ABSTRACTIVE_MODEL = "facebook/bart-large-cnn"
LLMLINGUA_MODEL = "microsoft/llmlingua-2-xlm-roberta-large-meetingbank"
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
SUMMAC_GRANULARITY = "sentence"


HYLAND_INTERACTIONAL = {
    "hedges": [
        "might", "may", "could", "possibly", "perhaps", "probably", "likely",
        "seem", "seems", "seemed", "appear", "appears", "generally", "usually",
        "often", "somewhat", "relatively", "quite", "rather", "roughly",
        "approximately", "tend to", "tends to", "in general", "to some extent",
        "presumably", "apparently", "arguably", "it is possible", "in most cases",
        "more or less", "in some ways", "a bit", "kind of", "sort of",
    ],
    "boosters": [
        "definitely", "certainly", "clearly", "obviously", "undoubtedly",
        "absolutely", "indeed", "in fact", "of course", "surely", "truly",
        "really", "extremely", "highly", "strongly", "always", "never",
        "without a doubt", "it is clear that", "there is no doubt",
    ],
    "attitude_markers": [
        "unfortunately", "fortunately", "surprisingly", "hopefully", "sadly",
        "remarkably", "understandably", "naturally", "interestingly",
        "i understand", "i'm sorry", "i am sorry", "sorry to hear",
        "don't worry", "do not worry", "no need to worry", "rest assured",
        "it's scary", "it is scary", "stressful", "overwhelming", "frightening",
        "upsetting", "distressing", "i know this is", "this must be",
        "that sounds", "completely normal", "anyone would", "it's understandable",
        "it is understandable", "you're not alone", "you are not alone",
        "difficult situation", "i can imagine", "how stressful", "how worrying",
    ],
    "engagement_markers": [
        "let's", "let us", "consider", "note that", "please note",
        "keep in mind", "bear in mind", "remember that", "imagine",
        "think about", "feel free", "don't hesitate", "do not hesitate",
        "take a deep breath", "take your time", "you may want to",
        "you might want to", "if you'd like", "if you would like",
        "you should know", "as you can see", "you'll want to",
    ],
    "self_mention": [
        "i can help", "i'd recommend", "i would recommend", "i suggest",
        "i'm here", "i am here", "let me", "we can", "i'd suggest",
        "in my view", "i think", "i believe", "i'd advise",
    ],
}

FLUFF_CATEGORIES = ["hedges", "attitude_markers", "engagement_markers", "self_mention"]

_LINE = re.compile(r"\n+")
_SENT = re.compile(r"(?<![0-9])(?<=[.!?])\s+(?=[^\s])")
_WORD = re.compile(r"\b\w+\b", re.UNICODE)


def split_sentences(text: str) -> List[str]:
    out = []
    for line in _LINE.split(text.strip()):
        line = line.strip()
        if not line:
            continue
        for s in _SENT.split(line):
            s = s.strip()
            if s:
                out.append(s)
    return out


def words(text: str) -> List[str]:
    return _WORD.findall(text.lower())


class TokenCounter:
    def __init__(self, name: str = COUNT_TOKENIZER):
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)

    def __call__(self, text: str) -> int:
        return len(self.tok.encode(text, add_special_tokens=False))

## STEP 1. 후보 생성 (TextRank / KL-Sum / LLMLingua-2)

In [ ]:
# ==============================================================
# STEP 1. 후보 생성 (TextRank / KL-Sum / LLMLingua-2)
# ==============================================================

def textrank_scores(sents: List[str], d: float = 0.85,
                    iters: int = 100, tol: float = 1e-6) -> np.ndarray:
    n = len(sents)
    if n == 0:
        return np.zeros(0)
    if n == 1:
        return np.ones(1)
    sets = [set(words(s)) for s in sents]
    lens = [len(words(s)) for s in sents]
    W = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j or lens[i] < 2 or lens[j] < 2:
                continue
            denom = math.log(lens[i]) + math.log(lens[j])
            if denom <= 0:
                continue
            W[i, j] = len(sets[i] & sets[j]) / denom
    out = W.sum(axis=1)
    scores = np.ones(n)
    for _ in range(iters):
        prev = scores.copy()
        for i in range(n):
            acc = 0.0
            for j in range(n):
                if W[j, i] > 0 and out[j] > 0:
                    acc += W[j, i] / out[j] * scores[j]
            scores[i] = (1 - d) + d * acc
        if np.abs(scores - prev).max() < tol:
            break
    return scores


def textrank_compress(text: str, ratio: float) -> str:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return text
    k = max(1, int(round(len(sents) * ratio)))
    scores = textrank_scores(sents)
    keep = sorted(np.argsort(-scores)[:k])
    return " ".join(sents[i] for i in keep)


def klsum_compress(text: str, ratio: float, eps: float = 1e-4) -> str:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return text
    doc_words = words(text)
    budget = max(1, int(round(len(doc_words) * ratio)))
    cnt = Counter(doc_words)
    total = sum(cnt.values())
    P = {w: c / total for w, c in cnt.items()}
    vocab = list(P.keys())

    def kl(tokens: List[str]) -> float:
        if not tokens:
            return float("inf")
        c = Counter(tokens)
        n = sum(c.values())
        acc = 0.0
        for w in vocab:
            q = c.get(w, 0) / n
            acc += P[w] * math.log(P[w] / max(q, eps))
        return acc

    chosen: List[int] = []
    chosen_words: List[str] = []
    remaining = list(range(len(sents)))
    while remaining:
        best, best_kl = None, float("inf")
        for i in remaining:
            t = words(sents[i])
            if len(chosen_words) + len(t) > budget:
                continue
            v = kl(chosen_words + t)
            if v < best_kl:
                best, best_kl = i, v
        if best is None:
            break
        chosen.append(best)
        chosen_words += words(sents[best])
        remaining.remove(best)
    if not chosen:
        chosen = [int(np.argmax(textrank_scores(sents)))]
    return " ".join(sents[i] for i in sorted(chosen))


class LLMLinguaCompressor:
    def __init__(self, model: str = LLMLINGUA_MODEL):
        from llmlingua import PromptCompressor
        self.c = PromptCompressor(model_name=model, use_llmlingua2=True,
                                  device_map=DEVICE)

    def __call__(self, text: str, rate: float) -> str:
        r = self.c.compress_prompt(
            text,
            rate=rate,
            force_tokens=["\n", ".", "!", "?", ",", ":", ";",
                          "not", "never", "no", "don't", "do", "n't"],
            force_reserve_digit=True,
            drop_consecutive=True,
        )
        return r["compressed_prompt"].strip()


class AbstractiveCompressor:
    def __init__(self, model: str = ABSTRACTIVE_MODEL):
        self.tok = AutoTokenizer.from_pretrained(model)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model).to(DEVICE).eval()

    @torch.no_grad()
    def __call__(self, text: str, ratio: float = 0.35) -> str:
        enc = self.tok([text], return_tensors="pt", truncation=True,
                       max_length=1024).to(DEVICE)
        n_in = int(enc["input_ids"].shape[1])
        target = max(16, int(n_in * ratio))
        ids = self.model.generate(
            **enc,
            num_beams=4,
            do_sample=False,
            min_length=max(8, int(target * 0.6)),
            max_length=max(20, int(target * 1.4)),
            length_penalty=1.0,
            no_repeat_ngram_size=3,
        )
        return self.tok.decode(ids[0], skip_special_tokens=True).strip()


@dataclass
class Candidate:
    text: str
    source: str
    param: float


class CandidateGenerator:
    def __init__(self, use_llmlingua: bool = True, use_abstractive: bool = True):
        self.lingua = LLMLinguaCompressor() if use_llmlingua else None
        self.abstractive = AbstractiveCompressor() if use_abstractive else None

    def __call__(self, text: str,
                 textrank_ratios: Tuple[float, ...] = (0.5, 0.65, 0.8),
                 klsum_ratios: Tuple[float, ...] = (0.4, 0.55, 0.7),
                 lingua_rates: Tuple[float, ...] = (0.4, 0.55, 0.7, 0.85),
                 abstractive_ratios: Tuple[float, ...] = (0.35, 0.5),
                 ) -> List[Candidate]:
        cands: List[Candidate] = []
        for r in textrank_ratios:
            cands.append(Candidate(textrank_compress(text, r), "textrank", r))
        for r in klsum_ratios:
            cands.append(Candidate(klsum_compress(text, r), "klsum", r))
        if self.lingua is not None:
            for r in lingua_rates:
                try:
                    cands.append(Candidate(self.lingua(text, r), "llmlingua2", r))
                except Exception:
                    continue
        if self.abstractive is not None:
            for r in abstractive_ratios:
                try:
                    cands.append(Candidate(self.abstractive(text, r), "abstractive", r))
                except Exception:
                    continue
        seen, uniq = set(), []
        for c in cands:
            key = c.text.strip().lower()
            if not key or key in seen:
                continue
            seen.add(key)
            uniq.append(c)
        return uniq

## STEP 2. Fluff 측정 (Hyland Metadiscourse)

In [ ]:
# ==============================================================
# STEP 2. Fluff 측정 (Hyland Metadiscourse)
# ==============================================================

class HylandMeter:
    def __init__(self, markers: Dict[str, List[str]] = None,
                 fluff_categories: List[str] = None):
        self.markers = markers or HYLAND_INTERACTIONAL
        self.categories = fluff_categories or FLUFF_CATEGORIES
        self.patterns = {
            cat: [re.compile(r"(?<!\w)" + re.escape(m) + r"(?!\w)")
                  for m in sorted(lst, key=len, reverse=True)]
            for cat, lst in self.markers.items()
        }

    def counts(self, text: str) -> Dict[str, float]:
        low = text.lower()
        n = len(words(text))
        out: Dict[str, float] = {}
        for cat, pats in self.patterns.items():
            out[cat] = sum(len(p.findall(low)) for p in pats)
        out["n_words"] = n
        out["fluff_markers"] = sum(out[c] for c in self.categories)
        out["md_density"] = 100.0 * out["fluff_markers"] / max(n, 1)
        out["yule_k"] = self.yule_k(text)
        return out

    @staticmethod
    def yule_k(text: str) -> float:
        w = words(text)
        n = len(w)
        if n == 0:
            return 0.0
        freq = Counter(Counter(w).values())
        s = sum(i * i * v for i, v in freq.items())
        return 1e4 * (s - n) / max(n * n, 1)

    def __call__(self, text: str) -> float:
        return self.counts(text)["md_density"]

## STEP 3. 정보 보존 (Nugget Evaluation)

In [ ]:
# ==============================================================
# STEP 3. 정보 보존 (Nugget Evaluation)
# ==============================================================

@dataclass
class NuggetSet:
    nuggets: List[str]
    vital: List[bool]
    allowance_c: Optional[float] = None

    def __post_init__(self):
        if self.allowance_c is None:
            lens = [len(n.replace(" ", "")) for n in self.nuggets]
            self.allowance_c = float(np.mean(lens)) if lens else 100.0


class NLIEngine:
    def __init__(self, model: str = NLI_MODEL, batch_size: int = 16):
        self.tok = AutoTokenizer.from_pretrained(model)
        self.model = AutoModelForSequenceClassification.from_pretrained(model)
        self.model = self.model.to(DEVICE).eval()
        self.batch_size = batch_size
        id2label = self.model.config.id2label
        self.entail_idx = next(
            i for i, l in id2label.items() if "entail" in str(l).lower()
        )

    @torch.no_grad()
    def entailment(self, premises: List[str], hypotheses: List[str]) -> np.ndarray:
        scores = []
        for i in range(0, len(premises), self.batch_size):
            p = premises[i:i + self.batch_size]
            h = hypotheses[i:i + self.batch_size]
            enc = self.tok(p, h, return_tensors="pt", truncation=True,
                           padding=True, max_length=512).to(DEVICE)
            logits = self.model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, self.entail_idx]
            scores.append(probs.detach().cpu().numpy())
        return np.concatenate(scores) if scores else np.zeros(0)


class NuggetEvaluator:
    def __init__(self, nli: NLIEngine, threshold: float = 0.5, beta: float = 3.0):
        self.nli = nli
        self.threshold = threshold
        self.beta = beta

    def assign(self, candidate: str, ns: NuggetSet) -> np.ndarray:
        prem = [candidate] * len(ns.nuggets)
        return self.nli.entailment(prem, ns.nuggets)

    def score(self, candidate: str, ns: NuggetSet) -> Dict[str, float]:
        probs = self.assign(candidate, ns)
        covered = probs >= self.threshold
        vital = np.array(ns.vital, dtype=bool)

        a = int(vital.sum())
        r = int((covered & vital).sum())
        R = int(covered.sum())

        recall_vital = r / a if a > 0 else 1.0
        recall_all = R / len(ns.nuggets) if ns.nuggets else 1.0

        length = len(candidate.replace(" ", "").replace("\n", ""))
        allowance = ns.allowance_c * R
        precision = 1.0 if length < allowance else 1.0 - (length - allowance) / max(length, 1)
        precision = max(precision, 0.0)

        b2 = self.beta ** 2
        denom = b2 * precision + recall_vital
        f = ((b2 + 1) * precision * recall_vital / denom) if denom > 0 else 0.0

        return {
            "recall_vital": recall_vital,
            "recall_all": recall_all,
            "precision": precision,
            "f_beta": f,
            "n_covered": R,
            "allowance": allowance,
            "length_nows": length,
            "covered_mask": covered.tolist(),
        }

## STEP 4. 사실성 검증 (SummaC / AlignScore)

In [ ]:
# ==============================================================
# STEP 4. 사실성 검증 (SummaC / AlignScore)
# ==============================================================

class FaithfulnessChecker:
    def __init__(self, use_summac: bool = True, use_alignscore: bool = False,
                 alignscore_ckpt: Optional[str] = None):
        self.summac = None
        self.align = None
        if use_summac:
            try:
                from summac.model_summac import SummaCConv
                self.summac = SummaCConv(
                    models=["vitc"], bins="percentile",
                    granularity=SUMMAC_GRANULARITY,
                    nli_labels="e", device=DEVICE, start_file="default",
                    agg="mean",
                )
            except Exception:
                self.summac = None
        if use_alignscore and alignscore_ckpt:
            try:
                from alignscore import AlignScore
                self.align = AlignScore(
                    model="roberta-base", batch_size=16, device=DEVICE,
                    ckpt_path=alignscore_ckpt, evaluation_mode="nli_sp",
                )
            except Exception:
                self.align = None

    def check(self, original: str, candidate: str) -> Dict[str, Optional[float]]:
        out: Dict[str, Optional[float]] = {
            "summac_consistency": None,
            "summac_retention": None,
            "align_consistency": None,
            "align_retention": None,
        }
        if self.summac is not None:
            out["summac_consistency"] = float(
                self.summac.score([original], [candidate])["scores"][0]
            )
            out["summac_retention"] = float(
                self.summac.score([candidate], [original])["scores"][0]
            )
        if self.align is not None:
            out["align_consistency"] = float(
                self.align.score(contexts=[original], claims=[candidate])[0]
            )
            out["align_retention"] = float(
                self.align.score(contexts=[candidate], claims=[original])[0]
            )
        return out

## 최종 선택 + Fluff Index 산출

In [ ]:
# ==============================================================
# 최종 선택 + Fluff Index 산출
# ==============================================================

@dataclass
class Constraints:
    recall_vital_min: float = 1.0
    recall_all_min: float = 0.90
    summac_consistency_min: float = 0.20
    summac_retention_min: float = 0.20
    min_token_ratio: float = 0.25


@dataclass
class Scored:
    candidate: Candidate
    fluff: float
    hyland: Dict[str, float]
    nugget: Dict[str, float]
    faith: Dict[str, Optional[float]]
    n_tokens: int
    feasible: bool
    violations: List[str] = field(default_factory=list)


class DeFluffPipeline:
    def __init__(self,
                 use_llmlingua: bool = True,
                 use_abstractive: bool = True,
                 use_summac: bool = True,
                 use_alignscore: bool = False,
                 alignscore_ckpt: Optional[str] = None,
                 constraints: Constraints = None):
        self.count = TokenCounter()
        self.generator = CandidateGenerator(use_llmlingua, use_abstractive)
        self.meter = HylandMeter()
        self.nli = NLIEngine()
        self.nuggeteer = NuggetEvaluator(self.nli)
        self.faith = FaithfulnessChecker(use_summac, use_alignscore, alignscore_ckpt)
        self.constraints = constraints or Constraints()

    def concise_fluff_index(self, text: str) -> Dict[str, float]:
        n0 = self.count(text)
        if n0 == 0:
            return {"fluff_index": 0.0, "p1": 1.0, "p2": 1.0, "p3": 1.0, "n0": 0}
        if self.generator.abstractive is not None:
            c1 = self.generator.abstractive(text, 0.35)
        else:
            c1 = klsum_compress(text, 0.35)
        c2 = klsum_compress(text, 0.5)
        if self.generator.lingua is not None:
            try:
                c3 = self.generator.lingua(text, 0.5)
            except Exception:
                c3 = textrank_compress(text, 0.5)
        else:
            c3 = textrank_compress(text, 0.5)
        p1 = self.count(c1) / n0
        p2 = self.count(c2) / n0
        p3 = self.count(c3) / n0
        return {
            "fluff_index": 1.0 - float(np.mean([p1, p2, p3])),
            "p1": p1, "p2": p2, "p3": p3, "n0": n0,
        }

    def _evaluate(self, original: str, cand: Candidate,
                  ns: NuggetSet, n0: int) -> Scored:
        hy = self.meter.counts(cand.text)
        ng = self.nuggeteer.score(cand.text, ns)
        fa = self.faith.check(original, cand.text)
        nt = self.count(cand.text)

        c = self.constraints
        v: List[str] = []
        if ng["recall_vital"] < c.recall_vital_min:
            v.append("recall_vital")
        if ng["recall_all"] < c.recall_all_min:
            v.append("recall_all")
        if fa["summac_consistency"] is not None and \
                fa["summac_consistency"] < c.summac_consistency_min:
            v.append("summac_consistency")
        if fa["summac_retention"] is not None and \
                fa["summac_retention"] < c.summac_retention_min:
            v.append("summac_retention")
        if nt < c.min_token_ratio * n0:
            v.append("min_length")

        return Scored(
            candidate=cand,
            fluff=hy["md_density"],
            hyland=hy,
            nugget=ng,
            faith=fa,
            n_tokens=nt,
            feasible=(len(v) == 0),
            violations=v,
        )

    def run(self, original: str, nuggets: List[str],
            vital: List[bool], allowance_c: Optional[float] = None) -> Dict:
        ns = NuggetSet(nuggets=nuggets, vital=vital, allowance_c=allowance_c)
        n0 = self.count(original)

        cands = self.generator(original)
        scored = [self._evaluate(original, c, ns, n0) for c in cands]
        baseline = self._evaluate(
            original, Candidate(original, "original", 1.0), ns, n0
        )

        feasible = [s for s in scored if s.feasible]
        chosen = min(feasible, key=lambda s: s.fluff) if feasible else baseline

        fi_orig = self.concise_fluff_index(original)
        fi_chosen = self.concise_fluff_index(chosen.candidate.text)

        return {
            "original": {
                "text": original,
                "n_tokens": n0,
                "md_density": baseline.fluff,
                "yule_k": baseline.hyland["yule_k"],
                "nugget": baseline.nugget,
                "fluff_index": fi_orig,
            },
            "chosen": {
                "text": chosen.candidate.text,
                "source": chosen.candidate.source,
                "param": chosen.candidate.param,
                "n_tokens": chosen.n_tokens,
                "md_density": chosen.fluff,
                "yule_k": chosen.hyland["yule_k"],
                "nugget": chosen.nugget,
                "faith": chosen.faith,
                "fluff_index": fi_chosen,
                "found_feasible": bool(feasible),
            },
            "delta": {
                "md_density": chosen.fluff - baseline.fluff,
                "fluff_index": fi_chosen["fluff_index"] - fi_orig["fluff_index"],
                "compression": chosen.n_tokens / max(n0, 1),
            },
            "candidates": [
                {
                    "source": s.candidate.source,
                    "param": s.candidate.param,
                    "n_tokens": s.n_tokens,
                    "md_density": round(s.fluff, 4),
                    "recall_vital": round(s.nugget["recall_vital"], 4),
                    "recall_all": round(s.nugget["recall_all"], 4),
                    "f_beta": round(s.nugget["f_beta"], 4),
                    "summac_consistency": s.faith["summac_consistency"],
                    "summac_retention": s.faith["summac_retention"],
                    "feasible": s.feasible,
                    "violations": s.violations,
                    "text": s.candidate.text,
                }
                for s in scored
            ],
        }

    def run_dataset(self, records: List[Dict], out_path: str = None) -> List[Dict]:
        results = []
        for i, rec in enumerate(records):
            r = self.run(
                original=rec["response"],
                nuggets=rec["nuggets"],
                vital=rec["vital"],
                allowance_c=rec.get("allowance_c"),
            )
            r["id"] = rec.get("id", i)
            r["persona"] = rec.get("persona")
            r["domain"] = rec.get("domain")
            results.append(r)
        if out_path:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
        return results

## DPO 페어 생성

In [ ]:
def build_dpo_pairs(results: List[Dict], prompts: Dict, min_gap: float = 0.05) -> List[Dict]:
    pairs = []
    for r in results:
        if not r["chosen"]["found_feasible"]:
            continue
        gap = r["original"]["md_density"] - r["chosen"]["md_density"]
        if gap < min_gap:
            continue
        pairs.append({
            "prompt": prompts.get(r["id"], ""),
            "chosen": r["chosen"]["text"],
            "rejected": r["original"]["text"],
            "meta": {
                "md_gap": gap,
                "fluff_index_gap": -r["delta"]["fluff_index"],
                "compression": r["delta"]["compression"],
                "source": r["chosen"]["source"],
                "recall_vital": r["chosen"]["nugget"]["recall_vital"],
            },
        })
    return pairs

## 금융 사기 시나리오용 Nugget 정의

In [ ]:
FINANCIAL_NUGGETS = [
    "Do not click the suspicious link or reply to the message.",
    "Contact the bank using the official phone number, not the number in the message.",
    "Check recent transaction history for unauthorized activity.",
    "Suspend or block the card if necessary.",
    "Report the incident to the police.",
    "Never share passwords, verification codes, ID photos, or account details.",
    "Change passwords for affected accounts.",
    "Register for a payment suspension or account freeze service.",
    "Keep evidence such as screenshots and call records.",
    "Ask a trusted person or a community center for help if needed.",
]

FINANCIAL_VITAL = [True, True, False, True, True, True, False, False, False, False]

## 데모 실행

In [ ]:
if __name__ == "__main__":
    demo = (
        "I'm so sorry to hear that this happened to you, and I completely understand "
        "how stressful and frightening this situation must feel right now. Please don't "
        "worry too much, because anyone would feel overwhelmed in this kind of situation, "
        "and it's completely normal to be upset. Let me help you work through this step "
        "by step. First, I would strongly recommend that you do not click on the "
        "suspicious link that you received, and you should never share your password, "
        "verification code, or account details with anyone. Second, you may want to "
        "contact your bank directly using the official phone number printed on the back "
        "of your card, rather than any number provided in the message itself. Third, "
        "it would probably be a good idea to check your recent transaction history "
        "carefully for any unauthorized activity, and if you notice anything unusual, "
        "you should suspend or block your card immediately. Finally, please remember to "
        "report the incident to the police, and rest assured that you are not alone in "
        "dealing with this."
    )

    pipe = DeFluffPipeline(use_llmlingua=True, use_abstractive=True, use_summac=True)
    result = pipe.run(demo, FINANCIAL_NUGGETS, FINANCIAL_VITAL)

    print(json.dumps({
        "original_md": round(result["original"]["md_density"], 3),
        "chosen_md": round(result["chosen"]["md_density"], 3),
        "original_FI": round(result["original"]["fluff_index"]["fluff_index"], 4),
        "chosen_FI": round(result["chosen"]["fluff_index"]["fluff_index"], 4),
        "source": result["chosen"]["source"],
        "compression": round(result["delta"]["compression"], 3),
        "recall_vital": result["chosen"]["nugget"]["recall_vital"],
        "n_feasible": sum(c["feasible"] for c in result["candidates"]),
        "n_candidates": len(result["candidates"]),
    }, indent=2, ensure_ascii=False))
    print()
    print(result["chosen"]["text"])